<a href="https://colab.research.google.com/github/Docdoi/AI-Engineered_Nanoproteins_A_De_Novo_Approach_to_Inhibit_Hantavirus_Transmission-/blob/main/rf/examples/diffusion_mutant_postfusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**RFdiffusion v1.1.1**
RFdiffusion is a method for structure generation, with or without conditional information (a motif, target etc). It can perform a whole range of protein design challenges as we have outlined in the RFdiffusion [manuscript](https://www.biorxiv.org/content/10.1101/2022.12.09.519842v2).


For **instructions**, see end of Notebook.

**<font color="red">NOTE:</font>**  This is tagged v1.1.1 of the notebook, this notebook may break in the future when colab updates. For latest version see [main](https://colab.research.google.com/github/sokrypton/ColabDesign/blob/main/rf/examples/diffusion.ipynb) branch.

Additional Notebooks:

- See [diffusion_foldcond](https://colab.research.google.com/github/sokrypton/ColabDesign/blob/v1.1.1/rf/examples/diffusion_foldcond.ipynb) for fold conditioning functionality.

- See [original version](https://colab.research.google.com/github/sokrypton/ColabDesign/blob/v1.1.1/rf/examples/diffusion_ori.ipynb) of this notebook (from 31Mar2023).


In [1]:
#@title setup **RFdiffusion** (~3min)
%%time
import os, time, signal
import sys, random, string, re
if not os.path.isdir("params"):
  os.system("apt-get install aria2")
  os.system("mkdir params")
  # send param download into background
  os.system("(\
  aria2c -q -x 16 https://files.ipd.uw.edu/krypton/schedules.zip; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt; \
  aria2c -q -x 16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt; \
  aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; \
  tar -xf alphafold_params_2022-12-06.tar -C params; \
  touch params/done.txt) &")

if not os.path.isdir("RFdiffusion"):
  print("installing RFdiffusion...")
  os.system("git clone https://github.com/sokrypton/RFdiffusion.git")
  os.system("pip install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
  os.system("pip install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
  # 17Mar2024: adding --no-dependencies to avoid installing nvidia-cuda-* dependencies
  # 25Aug2025: updating dgi install to work with latest pytorch
  os.system("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
  os.system("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
  os.system("cd RFdiffusion/env/SE3Transformer; pip install .")
  os.system("wget -qnc https://files.ipd.uw.edu/krypton/ananas")
  os.system("chmod +x ananas")

if not os.path.isdir("colabdesign"):
  print("installing ColabDesign...")
  os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git@v1.1.1")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")

if not os.path.isdir("RFdiffusion/models"):
  print("downloading RFdiffusion params...")
  os.system("mkdir RFdiffusion/models")
  models = ["Base_ckpt.pt","Complex_base_ckpt.pt"]
  for m in models:
    while os.path.isfile(f"{m}.aria2"):
      time.sleep(5)
  os.system(f"mv {' '.join(models)} RFdiffusion/models")
  os.system("unzip schedules.zip; rm schedules.zip")

if 'RFdiffusion' not in sys.path:
  os.environ["DGLBACKEND"] = "pytorch"
  sys.path.append('RFdiffusion')

from google.colab import files
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import ipywidgets as widgets
import py3Dmol

from inference.utils import parse_pdb
from colabdesign.rf.utils import get_ca
from colabdesign.rf.utils import fix_contigs, fix_partial_contigs, fix_pdb, sym_it
from colabdesign.shared.protein import pdb_to_string
from colabdesign.shared.plot import plot_pseudo_3D

def get_pdb(pdb_code=None):
  if pdb_code is None or pdb_code == "":
    upload_dict = files.upload()
    pdb_string = upload_dict[list(upload_dict.keys())[0]]
    with open("tmp.pdb","wb") as out: out.write(pdb_string)
    return "tmp.pdb"
  elif os.path.isfile(pdb_code):
    return pdb_code
  elif len(pdb_code) == 4:
    if not os.path.isfile(f"{pdb_code}.pdb1"):
      os.system(f"wget -qnc https://files.rcsb.org/download/{pdb_code}.pdb1.gz")
      os.system(f"gunzip {pdb_code}.pdb1.gz")
    return f"{pdb_code}.pdb1"
  else:
    os.system(f"wget -qnc https://alphafold.ebi.ac.uk/files/AF-{pdb_code}-F1-model_v3.pdb")
    return f"AF-{pdb_code}-F1-model_v3.pdb"

def run_ananas(pdb_str, path, sym=None):
  pdb_filename = f"outputs/{path}/ananas_input.pdb"
  out_filename = f"outputs/{path}/ananas.json"
  with open(pdb_filename,"w") as handle:
    handle.write(pdb_str)

  cmd = f"./ananas {pdb_filename} -u -j {out_filename}"
  if sym is None: os.system(cmd)
  else: os.system(f"{cmd} {sym}")

  # parse results
  try:
    out = json.loads(open(out_filename,"r").read())
    results,AU = out[0], out[-1]["AU"]
    group = AU["group"]
    chains = AU["chain names"]
    rmsd = results["Average_RMSD"]
    print(f"AnAnaS detected {group} symmetry at RMSD:{rmsd:.3}")

    C = np.array(results['transforms'][0]['CENTER'])
    A = [np.array(t["AXIS"]) for t in results['transforms']]

    # apply symmetry and filter to the asymmetric unit
    new_lines = []
    for line in pdb_str.split("\n"):
      if line.startswith("ATOM"):
        chain = line[21:22]
        if chain in chains:
          x = np.array([float(line[i:(i+8)]) for i in [30,38,46]])
          if group[0] == "c":
            x = sym_it(x,C,A[0])
          if group[0] == "d":
            x = sym_it(x,C,A[1],A[0])
          coord_str = "".join(["{:8.3f}".format(a) for a in x])
          new_lines.append(line[:30]+coord_str+line[54:])
      else:
        new_lines.append(line)
    return results, "\n".join(new_lines)

  except:
    return None, pdb_str

def run(command, steps, num_designs=1, visual="none"):

  def run_command_and_get_pid(command):
    pid_file = '/dev/shm/pid'
    os.system(f'nohup {command} > /dev/null & echo $! > {pid_file}')
    with open(pid_file, 'r') as f:
      pid = int(f.read().strip())
    os.remove(pid_file)
    return pid
  def is_process_running(pid):
    try:
      os.kill(pid, 0)
    except OSError:
      return False
    else:
      return True

  run_output = widgets.Output()
  progress = widgets.FloatProgress(min=0, max=1, description='running', bar_style='info')
  display(widgets.VBox([progress, run_output]))

  # clear previous run
  for n in range(steps):
    if os.path.isfile(f"/dev/shm/{n}.pdb"):
      os.remove(f"/dev/shm/{n}.pdb")

  pid = run_command_and_get_pid(command)
  try:
    fail = False
    for _ in range(num_designs):

      # for each step check if output generated
      for n in range(steps):
        wait = True
        while wait and not fail:
          time.sleep(0.1)
          if os.path.isfile(f"/dev/shm/{n}.pdb"):
            pdb_str = open(f"/dev/shm/{n}.pdb").read()
            if pdb_str[-3:] == "TER":
              wait = False
            elif not is_process_running(pid):
              fail = True
          elif not is_process_running(pid):
            fail = True

        if fail:
          progress.bar_style = 'danger'
          progress.description = "failed"
          break

        else:
          progress.value = (n+1) / steps
          if visual != "none":
            with run_output:
              run_output.clear_output(wait=True)
              if visual == "image":
                xyz, bfact = get_ca(f"/dev/shm/{n}.pdb", get_bfact=True)
                fig = plt.figure()
                fig.set_dpi(100);fig.set_figwidth(6);fig.set_figheight(6)
                ax1 = fig.add_subplot(111);ax1.set_xticks([]);ax1.set_yticks([])
                plot_pseudo_3D(xyz, c=bfact, cmin=0.5, cmax=0.9, ax=ax1)
                plt.show()
              if visual == "interactive":
                view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
                view.addModel(pdb_str,'pdb')
                view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':0.5,'max':0.9}}})
                view.zoomTo()
                view.show()
        if os.path.exists(f"/dev/shm/{n}.pdb"):
          os.remove(f"/dev/shm/{n}.pdb")
      if fail:
        progress.bar_style = 'danger'
        progress.description = "failed"
        break

    while is_process_running(pid):
      time.sleep(0.1)

  except KeyboardInterrupt:
    os.kill(pid, signal.SIGTERM)
    progress.bar_style = 'danger'
    progress.description = "stopped"

def run_diffusion(contigs, path, pdb=None, iterations=50,
                  symmetry="none", order=1, hotspot=None,
                  chains=None, add_potential=False,
                  num_designs=1, visual="none"):

  full_path = f"outputs/{path}"
  os.makedirs(full_path, exist_ok=True)
  opts = [f"inference.output_prefix={full_path}",
          f"inference.num_designs={num_designs}"]

  if chains == "": chains = None

  # determine symmetry type
  if symmetry in ["auto","cyclic","dihedral"]:
    if symmetry == "auto":
      sym, copies = None, 1
    else:
      sym, copies = {"cyclic":(f"c{order}",order),
                     "dihedral":(f"d{order}",order*2)}[symmetry]
  else:
    symmetry = None
    sym, copies = None, 1

  # determine mode
  contigs = contigs.replace(","," ").replace(":"," ").split()
  is_fixed, is_free = False, False
  fixed_chains = []
  for contig in contigs:
    for x in contig.split("/"):
      a = x.split("-")[0]
      if a[0].isalpha():
        is_fixed = True
        if a[0] not in fixed_chains:
          fixed_chains.append(a[0])
      if a.isnumeric():
        is_free = True
  if len(contigs) == 0 or not is_free:
    mode = "partial"
  elif is_fixed:
    mode = "fixed"
  else:
    mode = "free"

  # fix input contigs
  if mode in ["partial","fixed"]:
    pdb_str = pdb_to_string(get_pdb(pdb), chains=chains)
    if symmetry == "auto":
      a, pdb_str = run_ananas(pdb_str, path)
      if a is None:
        print(f'ERROR: no symmetry detected')
        symmetry = None
        sym, copies = None, 1
      else:
        if a["group"][0] == "c":
          symmetry = "cyclic"
          sym, copies = a["group"], int(a["group"][1:])
        elif a["group"][0] == "d":
          symmetry = "dihedral"
          sym, copies = a["group"], 2 * int(a["group"][1:])
        else:
          print(f'ERROR: the detected symmetry ({a["group"]}) not currently supported')
          symmetry = None
          sym, copies = None, 1

    elif mode == "fixed":
      pdb_str = pdb_to_string(pdb_str, chains=fixed_chains)

    pdb_filename = f"{full_path}/input.pdb"
    with open(pdb_filename, "w") as handle:
      handle.write(pdb_str)

    parsed_pdb = parse_pdb(pdb_filename)
    opts.append(f"inference.input_pdb={pdb_filename}")
    if mode in ["partial"]:
      iterations = int(80 * (iterations / 200))
      opts.append(f"diffuser.partial_T={iterations}")
      contigs = fix_partial_contigs(contigs, parsed_pdb)
    else:
      opts.append(f"diffuser.T={iterations}")
      contigs = fix_contigs(contigs, parsed_pdb)
  else:
    opts.append(f"diffuser.T={iterations}")
    parsed_pdb = None
    contigs = fix_contigs(contigs, parsed_pdb)

  if hotspot is not None and hotspot != "":
    opts.append(f"ppi.hotspot_res=[{hotspot}]")

  # setup symmetry
  if sym is not None:
    sym_opts = ["--config-name symmetry", f"inference.symmetry={sym}"]
    if add_potential:
      sym_opts += ["'potentials.guiding_potentials=[\"type:olig_contacts,weight_intra:1,weight_inter:0.1\"]'",
                   "potentials.olig_intra_all=True","potentials.olig_inter_all=True",
                   "potentials.guide_scale=2","potentials.guide_decay=quadratic"]
    opts = sym_opts + opts
    contigs = sum([contigs] * copies,[])

  opts.append(f"'contigmap.contigs=[{' '.join(contigs)}]'")
  opts += ["inference.dump_pdb=True","inference.dump_pdb_path='/dev/shm'"]

  print("mode:", mode)
  print("output:", full_path)
  print("contigs:", contigs)

  opts_str = " ".join(opts)
  cmd = f"./RFdiffusion/run_inference.py {opts_str}"
  print(cmd)

  # RUN
  run(cmd, iterations, num_designs, visual=visual)

  # fix pdbs
  for n in range(num_designs):
    pdbs = [f"outputs/traj/{path}_{n}_pX0_traj.pdb",
            f"outputs/traj/{path}_{n}_Xt-1_traj.pdb",
            f"{full_path}_{n}.pdb"]
    for pdb in pdbs:
      with open(pdb,"r") as handle: pdb_str = handle.read()
      with open(pdb,"w") as handle: handle.write(fix_pdb(pdb_str, contigs))

  return contigs, copies

installing RFdiffusion...
installing ColabDesign...
downloading RFdiffusion params...
CPU times: user 10 s, sys: 1.32 s, total: 11.3 s
Wall time: 2min 15s


In [3]:
%%time
# @title run **RFdiffusion** with Terminal Log (Motif RMSD)
import os, random, string

name = "andv_mutant_postfusion"

contigs = "B659-696/0 B703-950"
pdb_input = "/content/6y5wB.pdb"
iterations = 200

hotspot = "B739, B766, B900"
num_designs = 8

out_dir = f"outputs/{name}_run"
os.makedirs(out_dir, exist_ok=True)


!./RFdiffusion/run_inference.py \
  inference.output_prefix={out_dir}/design \
  inference.num_designs={num_designs} \
  inference.input_pdb={pdb_input} \
  diffuser.T={iterations} \
  ppi.hotspot_res="[{hotspot}]" \
  'contigmap.contigs=[{contigs}]' \
  inference.dump_pdb=False

/content/./RFdiffusion/run_inference.py:55: SyntaxWarning: invalid escape sequence '\d'
  m = re.match(".*_(\d+)\.pdb$", e)
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
[2026-08-19 14:12:30,096][inference.model_runners][INFO] - Reading checkpoint from /content/RFdiffusion/inference/../models/Complex_base_ckpt.pt
This is inf_conf.ckpt_path
/content/RFdiffusion/inference/../models/Complex_base_ckpt.pt
Assembling -model, -diffuser and -preprocess configs from checkpoint
USING MODEL CONFIG: self._conf[model][n_extra_block] = 4
USING MODEL CONFIG: self._conf[model][n_main_block] = 32
USING MODEL CONFIG: self._conf[model][n_ref_block] = 4
USING MODEL CONFIG: self._conf[model][d_msa] = 256
USING MODEL CONFIG: self._conf[model

In [6]:
#@title Display 3D structure
import py3Dmol
import ipywidgets as widgets
from IPython.display import display


path = "andv_mutant_postfusion_run"
num_designs = 8

def plot_pdb(num=0):

    pdb_path = f"outputs/{path}/design_{num}.pdb"

    try:
        with open(pdb_path, 'r') as f:
            pdb_str = f.read()

        view = py3Dmol.view(width=600, height=400)
        view.addModel(pdb_str, 'pdb')


        view.setStyle({'cartoon': {'colorscheme': 'chain'}})
        view.zoomTo()
        view.show()
    except FileNotFoundError:
        print(f" not found: {pdb_path}")


dropdown = widgets.Dropdown(
    options=[(f"design:{k}", k) for k in range(num_designs)],
    value=0,
    description='Design:'
)

output = widgets.Output()

def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        with output:
            output.clear_output(wait=True)
            plot_pdb(change['new'])

dropdown.observe(on_change)
display(dropdown)
with output:
    plot_pdb(0)
display(output)

Dropdown(description='Design:', options=(('design:0', 0), ('design:1', 1), ('design:2', 2), ('design:3', 3), (…

Output()

In [21]:
import os

# Path to BioPython's file handler in Python 3.12
bio_file_path = "/usr/local/lib/python3.12/dist-packages/Bio/file.py"

if os.path.exists(bio_file_path):
    with open(bio_file_path, "r") as f:
        code = f.read()

    # Patch es_handle to yield StringIO/text streams directly instead of passing them to open()
    old_code = "with open(handleish, mode, **kwargs) as fp:"
    new_code = (
        "if hasattr(handleish, 'read'):\n"
        "            yield handleish\n"
        "        else:\n"
        "            with open(handleish, mode, **kwargs) as fp:"
    )

    if old_code in code:
        code = code.replace(old_code, new_code)
        with open(bio_file_path, "w") as f:
            f.write(code)
        print("Successfully patched BioPython for Python 3.12 stream compatibility!")
    else:
        print("BioPython already patched or code structure varies.")

In [23]:
!find /content -type f -size 0 -print | head -50

/content/.config/config_sentinel
/content/.config/gce
/content/RFdiffusion/env/SE3Transformer/se3_transformer/__init__.py
/content/RFdiffusion/env/SE3Transformer/se3_transformer/runtime/__init__.py
/content/RFdiffusion/env/SE3Transformer/build/lib/se3_transformer/__init__.py
/content/RFdiffusion/env/SE3Transformer/build/lib/se3_transformer/runtime/__init__.py
/content/RFdiffusion/env/SE3Transformer/build/lib/tests/__init__.py
/content/RFdiffusion/env/SE3Transformer/tests/__init__.py
/content/params/done.txt


In [24]:
!find /content -type f \( -name "*.npz" -o -name "*params*" \) \
-exec ls -lh {} \; | head -100

-rw-r--r-- 1 root root 5.3G Aug 19 14:07 /content/alphafold_params_2022-12-06.tar
-rw-r--r-- 1 root root 7.6K Aug 19 14:06 /content/RFdiffusion/inference/sym_rots.npz
-rw-rw-r-- 1 995210 89939 355M Jul 19  2021 /content/params/params_model_3.npz
-rw-rw-r-- 1 995210 89939 355M Jul 19  2021 /content/params/params_model_4.npz
-rw-rw-r-- 1 995210 89939 356M Jul 19  2021 /content/params/params_model_2.npz
-rw-rw-r-- 1 995210 89939 355M Jul 19  2021 /content/params/params_model_5_ptm.npz
-rw-rw-r-- 1 995210 89939 356M Nov 22  2022 /content/params/params_model_2_multimer_v3.npz
-rw-rw-r-- 1 995210 89939 356M Nov 22  2022 /content/params/params_model_1_multimer_v3.npz
-rw-rw-r-- 1 995210 89939 356M Nov 22  2022 /content/params/params_model_4_multimer_v3.npz
-rw-rw-r-- 1 995210 89939 356M Jul 19  2021 /content/params/params_model_1_ptm.npz
-rw-rw-r-- 1 995210 89939 356M Jul 19  2021 /content/params/params_model_2_ptm.npz
-rw-rw-r-- 1 995210 89939 356M Nov 22  2022 /content/params/params_model_3

In [25]:
!find /content -type f -name "*.npz" -exec du -h {} \; | sort -h | head -30

8.0K	/content/RFdiffusion/inference/sym_rots.npz
355M	/content/params/params_model_3.npz
355M	/content/params/params_model_3_ptm.npz
355M	/content/params/params_model_4.npz
355M	/content/params/params_model_4_ptm.npz
355M	/content/params/params_model_5.npz
355M	/content/params/params_model_5_ptm.npz
356M	/content/params/params_model_1_multimer_v3.npz
356M	/content/params/params_model_1.npz
356M	/content/params/params_model_1_ptm.npz
356M	/content/params/params_model_2_multimer_v3.npz
356M	/content/params/params_model_2.npz
356M	/content/params/params_model_2_ptm.npz
356M	/content/params/params_model_3_multimer_v3.npz
356M	/content/params/params_model_4_multimer_v3.npz
356M	/content/params/params_model_5_multimer_v3.npz


In [27]:
import os
import numpy as np
import glob

files = sorted(glob.glob("/content/params/*.npz"))

print("Found:", len(files), "NPZ files\n")

for f in files:
    try:
        data = np.load(f)
        keys = data.files
        print(
            " OK:",
            os.path.basename(f),
            "| size:",
            round(os.path.getsize(f)/1024**2, 1), "MB",
            "| arrays:",
            len(keys)
        )
        data.close()

    except Exception as e:
        print(
            " BROKEN:",
            os.path.basename(f),
            "|",
            repr(e)
        )

Found: 15 NPZ files

 OK: params_model_1.npz | size: 355.8 MB | arrays: 336
 OK: params_model_1_multimer_v3.npz | size: 355.8 MB | arrays: 330
 OK: params_model_1_ptm.npz | size: 355.8 MB | arrays: 338
 OK: params_model_2.npz | size: 355.8 MB | arrays: 336
 OK: params_model_2_multimer_v3.npz | size: 355.8 MB | arrays: 330
 OK: params_model_2_ptm.npz | size: 355.8 MB | arrays: 338
 OK: params_model_3.npz | size: 354.5 MB | arrays: 265
 OK: params_model_3_multimer_v3.npz | size: 355.8 MB | arrays: 330
 OK: params_model_3_ptm.npz | size: 354.5 MB | arrays: 267
 OK: params_model_4.npz | size: 354.5 MB | arrays: 265
 OK: params_model_4_multimer_v3.npz | size: 355.8 MB | arrays: 330
 OK: params_model_4_ptm.npz | size: 354.5 MB | arrays: 267
 OK: params_model_5.npz | size: 354.5 MB | arrays: 265
 OK: params_model_5_multimer_v3.npz | size: 355.8 MB | arrays: 330
 OK: params_model_5_ptm.npz | size: 354.5 MB | arrays: 267


In [28]:
import os

print("Does /content/params exist?",
      os.path.isdir("/content/params"))

print("\nFiles:")
for x in sorted(os.listdir("/content/params")):
    print(x)

Does /content/params exist? True

Files:
LICENSE
done.txt
params_model_1.npz
params_model_1_multimer_v3.npz
params_model_1_ptm.npz
params_model_2.npz
params_model_2_multimer_v3.npz
params_model_2_ptm.npz
params_model_3.npz
params_model_3_multimer_v3.npz
params_model_3_ptm.npz
params_model_4.npz
params_model_4_multimer_v3.npz
params_model_4_ptm.npz
params_model_5.npz
params_model_5_multimer_v3.npz
params_model_5_ptm.npz


In [29]:
import numpy as np
import os

_original_np_load = np.load

def debug_np_load(file, *args, **kwargs):
    print("\n🔍 np.load is trying to open:")
    print(file)

    if isinstance(file, (str, os.PathLike)):
        print("   exists:", os.path.exists(file))
        if os.path.exists(file):
            print("   size:", os.path.getsize(file), "bytes")

    return _original_np_load(file, *args, **kwargs)

np.load = debug_np_load

print("✅ np.load debugger installed")

✅ np.load debugger installed


In [32]:
%%bash

echo "===== EMPTY FILES ====="

find /content \
  -type f \
  -size 0 \
  \( -name "*.npz" -o \
     -name "*.npy" -o \
     -name "*.pkl" -o \
     -name "*.pdb" -o \
     -name "*.json" \) \
  -print

===== EMPTY FILES =====


In [34]:
import os

path = "andv_mutant_postfusion_run"

for i in range(8):
    pdb_file = f"/content/outputs/{path}/design_{i}.pdb"

    if os.path.exists(pdb_file):
        print(
            f"✅ design_{i}.pdb",
            "| size =", os.path.getsize(pdb_file), "bytes"
        )
    else:
        print(f"❌ MISSING: {pdb_file}")

✅ design_0.pdb | size = 76648 bytes
✅ design_1.pdb | size = 76648 bytes
✅ design_2.pdb | size = 76648 bytes
✅ design_3.pdb | size = 76648 bytes
✅ design_4.pdb | size = 76648 bytes
✅ design_5.pdb | size = 76648 bytes
✅ design_6.pdb | size = 76648 bytes
✅ design_7.pdb | size = 76648 bytes


In [35]:
import os

os.makedirs("/content/debugpatch", exist_ok=True)

code = r'''
import numpy as np
import os

_original_np_load = np.load

def debug_np_load(file, *args, **kwargs):
    print("\n========== NP.LOAD DEBUG ==========", flush=True)

    try:
        print("object:", file, flush=True)
        print("type:", type(file), flush=True)

        if isinstance(file, (str, os.PathLike)):
            print("path:", os.fspath(file), flush=True)
            print("exists:", os.path.exists(file), flush=True)

            if os.path.exists(file):
                print("size:", os.path.getsize(file), "bytes", flush=True)

        elif hasattr(file, "name"):
            print("file object name:", file.name, flush=True)

            try:
                print("current position:", file.tell(), flush=True)
            except:
                pass

    except Exception as e:
        print("debug error:", e, flush=True)

    return _original_np_load(file, *args, **kwargs)

np.load = debug_np_load

print("✅ subprocess np.load debugger active", flush=True)
'''

with open("/content/debugpatch/sitecustomize.py", "w") as f:
    f.write(code)

print("✅ Debugger created")

✅ Debugger created


In [46]:
import inspect
from colabdesign.af.alphafold.model import utils

print("utils.py location:")
print(inspect.getfile(utils))

print("\nflat_params_to_haiku source:")
print(inspect.getsource(utils.flat_params_to_haiku))

utils.py location:
/content/colabdesign/af/alphafold/model/utils.py

flat_params_to_haiku source:
def flat_params_to_haiku(params, fuse=None):
  """Convert a dictionary of NumPy arrays to Haiku parameters."""
  P = {}
  for path, array in params.items():
    scope, name = path.split('//')
    if scope not in P:
      P[scope] = {}
    P[scope][name] = jnp.array(array)
  if fuse is not None:
    for a in ["evoformer_iteration",
              "extra_msa_stack",
              "template_embedding/single_template_embedding/template_embedding_iteration",
              "template_embedding/single_template_embedding/template_pair_stack/__layer_stack_no_state"]:
      for b in ["triangle_multiplication_incoming","triangle_multiplication_outgoing"]:
        k = f"alphafold/alphafold_iteration/evoformer/{a}/{b}"

        if fuse and f"{k}/center_layer_norm" in P:
          for c in ["gate","projection"]:
            L = P.pop(f"{k}/left_{c}")
            R = P.pop(f"{k}/right_{c}")
            P[f

In [48]:
%%bash

grep -R -n "np.load" /content/colabdesign/af | head -50

/content/colabdesign/af/alphafold/model/data.py:40:      params = np.load(io.BytesIO(f.read()), allow_pickle=False)


In [50]:
%%bash

sed -n '1,80p' /content/colabdesign/af/alphafold/model/data.py

# Copyright 2021 DeepMind Technologies Limited
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""Convenience functions for reading data."""

import io
import os
from typing import List
from colabdesign.af.alphafold.model import utils
import haiku as hk
import numpy as np
# Internal import (7716).


def casp_model_names(data_dir: str) -> List[str]:
  params = os.listdir(os.path.join(data_dir, 'params'))
  return [os.path.splitext(filename)[0] for filename in params]


def get_model_haiku_params(

In [51]:
import os

def check_model(model_name, data_dir):
    candidates = [
        os.path.join(data_dir, "params", f"params_{model_name}.npz"),
        os.path.join(data_dir, f"params_{model_name}.npz"),
        os.path.join(data_dir, "params", f"{model_name}.npz"),
        os.path.join(data_dir, f"{model_name}.npz"),
    ]

    print(f"\nMODEL: {model_name}")
    print(f"data_dir: {data_dir}")

    for p in candidates:
        if os.path.isfile(p):
            print("✅ FOUND:", p)
            print("   size:", os.path.getsize(p), "bytes")

            with open(p, "rb") as f:
                x = f.read()

            print("   bytes read:", len(x))
        else:
            print("❌ not found:", p)


models = [
    "model_1_multimer_v3",
    "model_2_multimer_v3",
    "model_3_multimer_v3",
    "model_4_multimer_v3",
    "model_5_multimer_v3",
]

for model in models:
    check_model(model, "/content")


MODEL: model_1_multimer_v3
data_dir: /content
✅ FOUND: /content/params/params_model_1_multimer_v3.npz
   size: 373043148 bytes
   bytes read: 373043148
❌ not found: /content/params_model_1_multimer_v3.npz
❌ not found: /content/params/model_1_multimer_v3.npz
❌ not found: /content/model_1_multimer_v3.npz

MODEL: model_2_multimer_v3
data_dir: /content
✅ FOUND: /content/params/params_model_2_multimer_v3.npz
   size: 373043148 bytes
   bytes read: 373043148
❌ not found: /content/params_model_2_multimer_v3.npz
❌ not found: /content/params/model_2_multimer_v3.npz
❌ not found: /content/model_2_multimer_v3.npz

MODEL: model_3_multimer_v3
data_dir: /content
✅ FOUND: /content/params/params_model_3_multimer_v3.npz
   size: 373043148 bytes
   bytes read: 373043148
❌ not found: /content/params_model_3_multimer_v3.npz
❌ not found: /content/params/model_3_multimer_v3.npz
❌ not found: /content/model_3_multimer_v3.npz

MODEL: model_4_multimer_v3
data_dir: /content
✅ FOUND: /content/params/params_model_

In [52]:
from colabdesign.af.alphafold.model import data

models = [
    "model_1_multimer_v3",
    "model_2_multimer_v3",
    "model_3_multimer_v3",
]

for model in models:
    print("\nTesting:", model)

    try:
        params = data.get_model_haiku_params(
            model_name=model,
            data_dir="/content"
        )
        print("✅ SUCCESS:", model)
        print("Number of parameter groups:", len(params))

    except Exception as e:
        print("❌ FAILED:", model)
        print(type(e).__name__, ":", e)


Testing: model_1_multimer_v3

🔍 np.load is trying to open:
✅ SUCCESS: model_1_multimer_v3
Number of parameter groups: 146

Testing: model_2_multimer_v3

🔍 np.load is trying to open:
✅ SUCCESS: model_2_multimer_v3
Number of parameter groups: 146

Testing: model_3_multimer_v3

🔍 np.load is trying to open:
✅ SUCCESS: model_3_multimer_v3
Number of parameter groups: 146


In [53]:
%%bash

grep -R -n "data_dir" /content/colabdesign/af/model.py \
                    /content/colabdesign/rf/designability_test.py \
                    /content/colabdesign/af/prep.py | head -80

/content/colabdesign/af/model.py:29:               data_dir=".", 
/content/colabdesign/af/model.py:120:      params = data.get_model_haiku_params(model_name=model_name, data_dir=data_dir, fuse=True)


In [54]:
%%bash

grep -R -n "mk_af_model" /content/colabdesign/rf/designability_test.py \
                       /content/colabdesign/af | head -50

/content/colabdesign/rf/designability_test.py:4:from colabdesign.af import mk_af_model
/content/colabdesign/rf/designability_test.py:93:    af_model = mk_af_model(protocol="binder",**flags)
/content/colabdesign/rf/designability_test.py:102:    af_model = mk_af_model(protocol="fixbb",
/content/colabdesign/rf/designability_test.py:115:    af_model = mk_af_model(protocol="fixbb",**flags)
/content/colabdesign/af/__init__.py:10:from colabdesign.af.model import mk_af_model
/content/colabdesign/af/__init__.py:13:mk_design_model = mk_afdesign_model = mk_af_model
/content/colabdesign/af/model.py:23:class mk_af_model(design_model, _af_inputs, _af_loss, _af_prep, _af_design, _af_utils):


grep: /content/colabdesign/af/__pycache__/model.cpython-312.pyc: binary file matches
grep: /content/colabdesign/af/__pycache__/__init__.cpython-312.pyc: binary file matches


In [55]:
from pathlib import Path

file = Path("/content/colabdesign/rf/designability_test.py")
text = file.read_text()

# Add data_dir="/content" to mk_af_model calls
text = text.replace(
    'mk_af_model(protocol="binder",**flags)',
    'mk_af_model(protocol="binder", data_dir="/content", **flags)'
)

text = text.replace(
    'mk_af_model(protocol="fixbb",**flags)',
    'mk_af_model(protocol="fixbb", data_dir="/content", **flags)'
)

text = text.replace(
    'mk_af_model(protocol="partial",**flags)',
    'mk_af_model(protocol="partial", data_dir="/content", **flags)'
)

file.write_text(text)

print("✅ Patched designability_test.py")

✅ Patched designability_test.py


In [56]:
%%bash
grep -n "mk_af_model" /content/colabdesign/rf/designability_test.py

4:from colabdesign.af import mk_af_model
93:    af_model = mk_af_model(protocol="binder", data_dir="/content", **flags)
102:    af_model = mk_af_model(protocol="fixbb",
115:    af_model = mk_af_model(protocol="fixbb", data_dir="/content", **flags)


In [57]:
%%time
#@title run ProteinMPNN & AlphaFold Validation
import os

path = "andv_mutant_postfusion_run"
contigs_str = "B659-696: 703-950"

for n in range(8):
    pdb_file = f"outputs/{path}/design_{n}.pdb"
    if os.path.exists(pdb_file):
        print(f"\n================ Evaluating {pdb_file} ================")
        opts = [
            f"--pdb={pdb_file}",
            f"--loc=outputs/{path}",
            f"--contig={contigs_str}",
            f"--copies=1",
            f"--num_seqs=8",
            f"--num_recycles=3",
            f"--rm_aa='C'",
            f"--mpnn_sampling_temp=0.1",
            f"--num_designs=1",
            f"--initial_guess",
            f"--use_multimer"
        ]
        !python colabdesign/rf/designability_test.py {" ".join(opts)}


================ Evaluating outputs/andv_mutant_postfusion_run/design_0.pdb ================
{'pdb':'outputs/andv_mutant_postfusion_run/design_0.pdb','loc':'outputs/andv_mutant_postfusion_run','contigs':'B659-696:','copies':1,'num_seqs':8,'initial_guess':False,'use_multimer':False,'num_recycles':3,'rm_aa':'C','num_designs':1,'mpnn_sampling_temp':0.1}
protocol=partial
running proteinMPNN...
Traceback (most recent call last):
  File "/content/colabdesign/rf/designability_test.py", line 198, in <module>
    main(sys.argv[1:])
  File "/content/colabdesign/rf/designability_test.py", line 136, in main
    af_model.prep_inputs(pdb_filename, **prep_flags)
  File "/usr/local/lib/python3.12/dist-packages/colabdesign/af/prep.py", line 63, in _prep_fixbb
    self._pdb = prep_pdb(pdb_filename, chain=chain, ignore_missing=ignore_missing,
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/colabdesign/af/prep.py", line 42

In [11]:
import os
print(os.path.getsize("outputs/andv_mutant_postfusion_run/design_0.pdb"))

76648


In [39]:
%%time
#@title run **ProteinMPNN** to generate a sequence and **AlphaFold** to validate
import os, time

path = "andv_mutant_postfusion_run"
target_pdb = "/content/6y5wB.pdb"
contigs_str = "B659-696/0 B703-950"

num_seqs = 8
initial_guess = True
num_recycles = 3
use_multimer = True
rm_aa = "C"
mpnn_sampling_temp = 0.1
num_designs = 8

opts = [
    f"--pdb={target_pdb}",
    f"--loc=outputs/{path}",
    f"--contig={contigs_str}",
    f"--copies=1",
    f"--num_seqs={num_seqs}",
    f"--num_recycles={num_recycles}",
    f"--rm_aa='{rm_aa}'",
    f"--mpnn_sampling_temp={mpnn_sampling_temp}",
    f"--num_designs={num_designs}"
]

if initial_guess: opts.append("--initial_guess")
if use_multimer: opts.append("--use_multimer")

!python colabdesign/rf/designability_test.py {" ".join(opts)}

{'pdb':'/content/6y5wB.pdb','loc':'outputs/andv_mutant_postfusion_run','contigs':'B659-696/0','copies':1,'num_seqs':8,'initial_guess':False,'use_multimer':False,'num_recycles':3,'rm_aa':'C','num_designs':1,'mpnn_sampling_temp':0.1}
protocol=partial
running proteinMPNN...
Traceback (most recent call last):
  File "/content/colabdesign/rf/designability_test.py", line 198, in <module>
    main(sys.argv[1:])
  File "/content/colabdesign/rf/designability_test.py", line 136, in main
    af_model.prep_inputs(pdb_filename, **prep_flags)
  File "/usr/local/lib/python3.12/dist-packages/colabdesign/af/prep.py", line 63, in _prep_fixbb
    self._pdb = prep_pdb(pdb_filename, chain=chain, ignore_missing=ignore_missing,
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/colabdesign/af/prep.py", line 421, in prep_pdb
    protein_obj = protein.from_pdb_string(pdb_str, chain_id=chain)
                  ^^^^^^^^^^^^^^^^^^^^^^

In [ ]:
#@title Package and download results
#@markdown If you are having issues downloading the result archive,
#@markdown try disabling your adblocker and run this cell again.
#@markdown  If that fails click on the little folder icon to the
#@markdown  left, navigate to file: `name.result.zip`,
#@markdown  right-click and select \"Download\"
#@markdown (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).
!zip -r {path}.result.zip outputs/{path}* outputs/traj/{path}*
files.download(f"{path}.result.zip")

  adding: outputs/andv_run2_0.pdb (deflated 75%)
  adding: outputs/andv_run2_0.trb (deflated 25%)
  adding: outputs/andv_run2_1.pdb (deflated 76%)
  adding: outputs/andv_run2_1.trb (deflated 25%)
  adding: outputs/andv_run2_2.pdb (deflated 76%)
  adding: outputs/andv_run2_2.trb (deflated 26%)
  adding: outputs/andv_run2_3.pdb (deflated 75%)
  adding: outputs/andv_run2_3.trb (deflated 25%)
  adding: outputs/andv_run2_4.pdb (deflated 76%)
  adding: outputs/andv_run2_4.trb (deflated 24%)
  adding: outputs/andv_run2_5.pdb (deflated 76%)
  adding: outputs/andv_run2_5.trb (deflated 25%)
  adding: outputs/andv_run2_6.pdb (deflated 76%)
  adding: outputs/andv_run2_6.trb (deflated 25%)
  adding: outputs/andv_run2_7.pdb (deflated 76%)
  adding: outputs/andv_run2_7.trb (deflated 25%)
  adding: outputs/traj/andv_run2_0_pX0_traj.pdb (deflated 79%)
  adding: outputs/traj/andv_run2_0_Xt-1_traj.pdb (deflated 86%)
  adding: outputs/traj/andv_run2_1_pX0_traj.pdb (deflated 76%)
  adding: outputs/traj/and

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Instructions**
---
---

Use `contigs` to define continious chains. Use a `:` to define multiple contigs and a `/` to define mutliple segments within a contig.
For example:

**unconditional**
- `contigs='100'` - diffuse **monomer** of length 100
- `contigs='50:100'` - diffuse **hetero-oligomer** of lengths 50 and 100
- `contigs='50'` `symmetry='cyclic'` `order=2` - make two copies of the defined contig(s) and add a symmetry constraint, for **homo-oligomeric** diffusion.

**binder design**
- `contigs='A:50'` `pdb='4N5T'` - diffuse a **binder** of length 50 to chain A of defined PDB.
- `contigs='E6-155:70-100'` `pdb='5KQV'` `hotspot='E64,E88,E96'` - diffuse a **binder** of length 70 to 100 (sampled randomly) to chain E and defined hotspot(s).

**motif scaffolding**
 - `contigs='40/A163-181/40'` `pdb='5TPN'`
 - `contigs='A3-30/36/A33-68'` `pdb='6MRR'` - diffuse a loop of length 36 between two segments of defined PDB ranges.

**partial diffusion**
- `contigs=''` `pdb='6MRR'` - noise all coordinates
- `contigs='A1-10'` `pdb='6MRR'` - keep first 10 positions fixed, noise the rest
- `contigs='A'` `pdb='1SSC'` - fix chain A, noise the rest

*hints and tips*
- `pdb=''` leave blank to get an upload prompt
- `contigs='50-100'` use dash to specify a range of lengths to sample from